### Formula used
I tried to use the formula in the pdf, although there it is too granular and I do not have the "stage" variable for instance. Therefore I adjested it trying to maintain a the correct order of competitions' relevance. For the same reason I did not implemented the last point of the formula description about knock-out rounds. I hope it did not have much impact as I computed rolling elo not historical elo. 

In [221]:
import pandas as pd
import numpy as np

In [222]:
# importing the dataset
path = r"data\results.csv"
with open(path) as file:
    results = pd.read_csv(file) 

results.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [223]:
# Hardcoding Quarter-finals
results.loc[results.index[-3:], 'home_score'] = [2, 1, 3] 
results.loc[results.index[-3:], 'away_score'] = [1, 2, 1]
results.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49500,2026-07-07,Switzerland,Colombia,0.0,0.0,FIFA World Cup,Vancouver,Canada,True
49501,2026-07-09,France,Morocco,2.0,0.0,FIFA World Cup,Foxborough,United States,True
49502,2026-07-10,Spain,Belgium,2.0,1.0,FIFA World Cup,Inglewood,United States,True
49503,2026-07-11,Norway,England,1.0,2.0,FIFA World Cup,Miami Gardens,United States,True
49504,2026-07-12,Argentina,Switzerland,3.0,1.0,FIFA World Cup,Kansas City,United States,True


In [224]:
# considering only columns after 2002 World Cup and dropping irrelevant columns
results = results[results["date"] > "2002-06-30"] # 2002 World cup final date
results.drop(columns = ["city", "country", "neutral"], inplace=True)
results.head()

,date,home_team,away_team,home_score,away_score,tournament
26508,2002-07-03,Estonia,Azerbaijan,0.0,0.0,Friendly
26509,2002-07-05,New Zealand,Tahiti,4.0,0.0,Oceania Nations Cup
26510,2002-07-05,Papua New Guinea,Solomon Islands,0.0,0.0,Oceania Nations Cup
26511,2002-07-06,Australia,Vanuatu,2.0,0.0,Oceania Nations Cup
26512,2002-07-06,British Virgin Islands,Anguilla,2.0,1.0,Friendly


In [225]:
# sorting for "date" (just to be sure) and resetting index
results = results.sort_values("date").reset_index(drop=True)

In [226]:
# adding columns "winner", "shootouts" necessary to compute rolling elo
# importing an other dataset which contains shootouts data
path = r"data\shootouts.csv"
with open(path) as file:
    shootouts = pd.read_csv(file)

# Considering only world cups from 2002 World Cup final
shootouts = shootouts[shootouts["date"] > "2002-06-30"]

# dropping irrelevant columns
shootouts.drop(columns="first_shooter", inplace=True)

In [227]:
# Merging
df = pd.merge(results, shootouts, left_on=["date", "home_team", "away_team"], right_on=["date", "home_team", "away_team"], how="left")
df.head()

,date,home_team,away_team,home_score,away_score,tournament,winner
0,2002-07-03,Estonia,Azerbaijan,0.0,0.0,Friendly,NaN
1,2002-07-05,New Zealand,Tahiti,4.0,0.0,Oceania Nations Cup,NaN
2,2002-07-05,Papua New Guinea,Solomon Islands,0.0,0.0,Oceania Nations Cup,NaN
3,2002-07-06,Australia,Vanuatu,2.0,0.0,Oceania Nations Cup,NaN
4,2002-07-06,British Virgin Islands,Anguilla,2.0,1.0,Friendly,NaN


In [228]:
# testing if the merge was done correctly
df[:][df["date"] == "2026-06-29"]
# ok it is correct

,date,home_team,away_team,home_score,away_score,tournament,winner
22970,2026-06-29,Netherlands,Morocco,1.0,1.0,FIFA World Cup,Morocco
22971,2026-06-29,Brazil,Japan,2.0,1.0,FIFA World Cup,NaN
22972,2026-06-29,Germany,Paraguay,1.0,1.0,FIFA World Cup,Paraguay


In [229]:
# now let's add a dummy feature to encode if the match ended up with shootout
df["shootouts"] = 0
for i in range(df.shape[0]):
    if df.iloc[i, 3] == df.iloc[i, 4] and not pd.isna(df.iloc[i, -2]): # -2: "result"
        # the two scores are even and it is a knockout
        df.iloc[i, -1] = 1 # match to shootout
    else:
        pass
df.head()

,date,home_team,away_team,home_score,away_score,tournament,winner,shootouts
0,2002-07-03,Estonia,Azerbaijan,0.0,0.0,Friendly,NaN,0
1,2002-07-05,New Zealand,Tahiti,4.0,0.0,Oceania Nations Cup,NaN,0
2,2002-07-05,Papua New Guinea,Solomon Islands,0.0,0.0,Oceania Nations Cup,NaN,0
3,2002-07-06,Australia,Vanuatu,2.0,0.0,Oceania Nations Cup,NaN,0
4,2002-07-06,British Virgin Islands,Anguilla,2.0,1.0,Friendly,NaN,0


In [230]:
# testing if it is correct
df[:][df["date"] == "2026-06-29"]

,date,home_team,away_team,home_score,away_score,tournament,winner,shootouts
22970,2026-06-29,Netherlands,Morocco,1.0,1.0,FIFA World Cup,Morocco,1
22971,2026-06-29,Brazil,Japan,2.0,1.0,FIFA World Cup,NaN,0
22972,2026-06-29,Germany,Paraguay,1.0,1.0,FIFA World Cup,Paraguay,1


In [231]:
# managing "winner" for other matches ended in 90 mins
for i in range(df.shape[0]):
    if pd.isnull(df.iloc[i, -2]): # match did not finished with shootout
        if df.iloc[i, 3] > df.iloc[i, 4]: # first team beated the second
            df.iloc[i, -2] = df.iloc[i, 1] # home_team wins
        elif df.iloc[i, 3] < df.iloc[i, 4]:
            df.iloc[i, -2] = df.iloc[i, 2] # away_team wins
        else:
            df.iloc[i, -2] = "Draw" # draw
    else:
        pass
df.tail(15)

,date,home_team,away_team,home_score,away_score,tournament,winner,shootouts
22982,2026-07-03,Australia,Egypt,1.0,1.0,FIFA World Cup,Egypt,1
22983,2026-07-03,Argentina,Cape Verde,3.0,2.0,FIFA World Cup,Argentina,0
22984,2026-07-03,Colombia,Ghana,1.0,0.0,FIFA World Cup,Colombia,0
22985,2026-07-04,Canada,Morocco,0.0,3.0,FIFA World Cup,Morocco,0
22986,2026-07-04,Paraguay,France,0.0,1.0,FIFA World Cup,France,0
22987,2026-07-05,Brazil,Norway,1.0,2.0,FIFA World Cup,Norway,0
22988,2026-07-05,Mexico,England,2.0,3.0,FIFA World Cup,England,0
22989,2026-07-06,Portugal,Spain,0.0,1.0,FIFA World Cup,Spain,0
22990,2026-07-06,United States,Belgium,1.0,4.0,FIFA World Cup,Belgium,0
22991,2026-07-07,Argentina,Egypt,3.0,2.0,FIFA World Cup,Argentina,0


In [232]:
# checking dataset
df.head()

,date,home_team,away_team,home_score,away_score,tournament,winner,shootouts
0,2002-07-03,Estonia,Azerbaijan,0.0,0.0,Friendly,Draw,0
1,2002-07-05,New Zealand,Tahiti,4.0,0.0,Oceania Nations Cup,New Zealand,0
2,2002-07-05,Papua New Guinea,Solomon Islands,0.0,0.0,Oceania Nations Cup,Draw,0
3,2002-07-06,Australia,Vanuatu,2.0,0.0,Oceania Nations Cup,Australia,0
4,2002-07-06,British Virgin Islands,Anguilla,2.0,1.0,Friendly,British Virgin Islands,0


In [233]:
len(df["tournament"].unique())
# there are 119 kinds of tournaments
# let's classify them in a bunch of classes 
# similar to the ones provided by the FIFA document

119

In [234]:
# function to classify every kind of tournament 
def classify_tournament(tournament):
    t = tournament.lower()

    if t == "fifa world cup":
        return "FIFA World Cup final competition matches", 55 # mean between 50 and 60 (see FIFA ranking formula.pdf)
    
    if any(x in t for x in ["qualification", "qualifying", "q", "qual"]) or t.endswith("q"):
        # check if it is a European match
        if "uefa" in t or "euro" in t:
            return "European qualification matches for Confederations final competitions and for FIFA World Cup final competitions", 30
        else:
            return "Non-European qualification matches for Confederations final competitions and for FIFA World Cup final competitions", 25
    
    if "nations league" in t or "league" in t:
        # check if it is a European match
        if "uefa" in t or "euro" in t:
            return "European league competitions", 25
        else:
            return "Non-European league competitions", 20 # mean between 15 and 25 (see FIFA ranking formula.pdf)       
    
    continental_tournaments = [
        'copa américa', 'copa america', 'afc asian cup', 
        'african cup of nations', 'gold cup', 'oceania nations cup', 
        'conmebol–uefa cup of champions', 'cafa nations cup',
        'asean championship', 'aff championship'
    ]
    if t == "confederations cup" or any(tc in t for tc in continental_tournaments):
        return "Non-European confederation final competition matches; all FIFA Confederations Cup matches", 35
        # mean between 35 and 40 (see FIFA ranking formula.pdf)

    if 'uefa euro' in t:
        return "European confederation final competition matches; all FIFA Confederations Cup matches", 40
    
    return "Friendly and similar matches", 7.5 # mean between 5 and 10 (see FIFA ranking formula.pdf)


In [235]:
# function to check if the previous function correctly classified all kinds of tournaments
def check_tournaments_coverage(tournaments_list):
    classes = [
        'Friendly and similar matches',
        'Non-European league competitions',
        'European league competitions',
        'European qualification matches for Confederations final competitions and for FIFA World Cup final competitions',
        'Non-European qualification matches for Confederations final competitions and for FIFA World Cup final competitions',
        'Non-European confederation final competition matches; all FIFA Confederations Cup matches',
        'European confederation final competition matches; all FIFA Confederations Cup matches',
        'FIFA World Cup final competition matches'
    ]

    report = {i: [] for i in classes} # "i" could be thinked as "class"

    for tournament in tournaments_list:
        i, _ = classify_tournament(tournament)
        report[i].append(tournament)

    total_classified = 0
    for i, tournaments in report.items():
        print(f"{i} (I = {classify_tournament(tournaments[0])[1] if tournaments else "N/D"}):")
        print(f" Count: {len(tournaments)} tournaments")

        if tournaments:
            print(f" Examples: {tournaments[:8]} ... " if len(tournaments) > 8 else f"Content: {tournaments}")
        else:
            print("No torunament classified in this class")
        print("-" * 74)
        total_classified += len(tournaments)

    print(f"Verification completed: {total_classified} out of {len(tournaments_list)} correctly classified")

tournaments_list = list(df["tournament"].unique())
check_tournaments_coverage(tournaments_list)

Friendly and similar matches (I = 7.5):
 Count: 86 tournaments
 Examples: ['Friendly', 'COSAFA Cup', 'SKN Football Festival', 'CECAFA Cup', "Prime Minister's Cup", 'Arab Cup', 'SAFF Cup', 'Lunar New Year Cup'] ... 
--------------------------------------------------------------------------
Non-European league competitions (I = 20):
 Count: 1 tournaments
Content: ['CONCACAF Nations League']
--------------------------------------------------------------------------
European league competitions (I = 25):
 Count: 1 tournaments
Content: ['UEFA Nations League']
--------------------------------------------------------------------------
European qualification matches for Confederations final competitions and for FIFA World Cup final competitions (I = 30):
 Count: 1 tournaments
Content: ['UEFA Euro qualification']
--------------------------------------------------------------------------
Non-European qualification matches for Confederations final competitions and for FIFA World Cup final competi

In [236]:
final_dates = [
    "2002-06-30",
    "2006-07-09",
    "2010-07-11",
    "2014-07-13",
    "2018-07-14",
    "2022-12-18",
]
df["window"] = np.searchsorted(final_dates, df["date"])

In [237]:
# defining the formula to compute rolling elo
def compute_elo_rolling(window_group):
    current_elo = {}
    elo_home_list = []
    elo_away_list = []

    for idx, row in window_group.iterrows():
        home = row["home_team"]
        away = row["away_team"]

        # team with no matches in the new window, set its elo to 1500
        r_home = current_elo.get(home, 1500) # it returns the value associated to the key "home", if "home" is not in the dict returns "1500"
        r_away = current_elo.get(away, 1500)

        elo_home_list.append(r_home)
        elo_away_list.append(r_away)

        # elo's formula for teams with at least one match in the new window
        dr_home = r_home - r_away 
        dr_away = r_away - r_home

        e_home = 1 / (10**(-dr_home/600) + 1)
        e_away = 1 / (10**(-dr_away/600) + 1)

        # get I
        _, k = classify_tournament(row["tournament"]) # k = I (see "FIFA ranking formula.pdf")

        # match result
        if row.get("shootouts") == 1:
            if row["winner"] == home:
                w_home, w_away = 0.75, 0.5
            else:
                w_home, w_away = 0.5, 0.75
        else:
            if row["home_score"] > row["away_score"]:
                w_home, w_away = 1.0, 0.0
            elif row["home_score"] < row["away_score"]:
                w_home, w_away = 0.0, 1.0
            else:
                w_home, w_away = 0.5, 0.5
        
        # computing elo
        current_elo[home] = round(r_home + k *(w_home - e_home), 2)
        current_elo[away] = round(r_away + k *(w_away - e_away))

    # adding columns
    window_group["elo_home"] = elo_home_list
    window_group["elo_away"] = elo_away_list
    return window_group

In [238]:
df = df.groupby("window", group_keys=False).apply(compute_elo_rolling, include_groups = False)

In [ ]:
# checking if it is plausable
df.tail(50)

In [240]:
# attaching these 2 new columns to "dataset"
path = r"data\dataset.csv"
with open(path) as file:
    dataset = pd.read_csv(file)

In [241]:
# adding elo columns
elo = df[["elo_home", "elo_away"]]
dataset["elo_home"] = elo["elo_home"]
dataset["elo_away"] = elo["elo_away"]

In [242]:
# checking
dataset.tail(50)

,year,month,day,home_team,away_team,stage,result,shootouts,home_score,home_is_host,...,away_total_market_value_eur,away_avg_age,away_wc_participations_before,away_groups_passed_before,away_round16_before,away_quarterfinals_before,away_semifinals_before,away_finals_before,elo_home,elo_away
370,2026,6,24,Canada,Switzerland,0,1.0,0,1.0,1,...,2.510000e+08,27.5,12,8,7,3,0,0,1475.00,1463.00
371,2026,6,24,Bosnia and Herzegovina,Qatar,0,0.0,0,3.0,0,...,1.800000e+07,27.8,1,0,0,0,0,0,1503.42,1503.44
372,2026,6,24,Scotland,Brazil,0,1.0,0,0.0,0,...,9.320000e+08,27.8,22,20,12,17,11,6,1499.00,1496.25
373,2026,6,24,Morocco,Haiti,0,0.0,0,4.0,0,...,3.875000e+07,27.6,1,0,0,0,0,0,1533.00,1466.00
374,2026,6,25,United States,Turkey,0,1.0,0,2.0,1,...,5.082000e+08,27.0,2,1,1,1,1,0,1532.66,1533.00
375,2026,6,25,Paraguay,Australia,0,2.0,0,0.0,0,...,3.285000e+07,27.5,6,2,2,0,0,0,1517.36,1444.00
376,2026,6,25,Curaçao,Ivory Coast,0,1.0,0,0.0,0,...,3.708000e+08,26.6,3,0,0,0,0,0,1503.77,1550.41
377,2026,6,25,Ecuador,Germany,0,0.0,0,2.0,0,...,8.280000e+08,26.6,20,18,10,16,13,8,1496.00,1462.61
378,2026,6,25,Japan,Sweden,0,2.0,0,1.0,0,...,4.959800e+08,27.1,12,9,5,6,4,1,1478.66,1506.94
379,2026,6,25,Tunisia,Netherlands,0,1.0,0,1.0,0,...,8.080000e+08,26.6,11,11,9,7,5,3,1459.00,1492.00


In [243]:
dataset.to_csv(r"data\dataset_elo.csv", index=False)